# Emotion Recognition from Text (NLP)

This notebook demonstrates the development of a **deep learning model** that predicts human emotions from short text sentences.  

We use a **Bidirectional LSTM (BiLSTM)** architecture to classify sentences into six emotion categories: Joy, Sadness, Anger, Fear, Love, and Surprise. The dataset contains over 420,000 labeled samples.  

The workflow includes:
- Data loading and preprocessing
- Text tokenization and padding
- Model building and training
- Evaluation and prediction


Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.python.keras.layers import Layer, Dense
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

Load the dataset

In [ ]:
# load initial dataset
df = pd.read_csv('data/emotions.csv')
# display shape of the dataset
print(df.shape)
# display information about the dataset
df.info()
# display first few rows of the dataframe
df.head()

Clean & normalize the dataset

In [ ]:
# drop rows with missing values
df.dropna(inplace=True)
print("missing values per column:")
print(df.isnull().sum())

# remove duplicates
print("number of duplicate rows:",df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("number of duplicate rows after cleaning:", df.duplicated().sum())

# ensure correct data types
df['sentence'] = df['sentence'].astype('string')
df['emotion'] = df['emotion'].astype('string')
print("column types:")
print(df.dtypes)

# normalize the text input data
df['sentence'] = df['sentence'].str.lower().str.strip()

Exploratory data analysis

1. Emotion distribution

In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(x='emotion', data=df, palette='viridis', hue='emotion', legend=False)
plt.title("Emotion Distribution")
plt.xticks(rotation=45)
plt.show()

2. Sentence length distribution

In [ ]:
df['sentence_length'] = df['sentence'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10,4))
sns.histplot(df['sentence_length'], bins=50, kde=True, color='green')
plt.xlim(0, 100)
plt.title("Sentence Length Distribution")
plt.xlabel("Number of Words")
plt.show()

Label Encoding

In [ ]:
# encode target label
label_encoder = LabelEncoder()
df['emotion_encoded'] = label_encoder.fit_transform(df['emotion'])
print("classes:", label_encoder.classes_)

Split data into training and testing subsets

In [ ]:
# train-test split (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    df['sentence'], df['emotion_encoded'], 
    test_size=0.2, random_state=42, stratify=df['emotion_encoded']
)

Text Preprocessing (Tokenization & Padding)

In [ ]:
# vocab size (memorize the 20000 most common words)
max_words = 20000  
# max sequence length (maximum number of words in a sentence since neural networks need all inputs to be the same length.)
max_len = 60      

# tokenization
tokenizer = keras.preprocessing.text.Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# convert texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# pad sequences (add zeros to make them equal length, or truncate if text exceeds max length)
X_train_pad = keras.preprocessing.sequence.pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = keras.preprocessing.sequence.pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# convert to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)
num_classes = len(label_encoder.classes_)

Building a Bidirectional LSTM (BiLSTM) Model with Dropout

In [ ]:
# build the BiLSTM model
model_bilstm = tf.keras.models.Sequential([
    # layer 1: embedding Layer
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=128),
    # layer 2: bi-directional LSTM 
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(128, return_sequences=False, dropout=0.1, recurrent_dropout=0.1)
    ),
    # layer 3: dropout 
    tf.keras.layers.Dropout(0.1),
    # layer 4: dense layer (hidden layer)
    tf.keras.layers.Dense(128, activation='relu'),
    # layer 5: dropout 
    tf.keras.layers.Dropout(0.1),
    # layer 6: output layer ('softmax' activation is necessary for multi-class classification)
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# compile the model ('sparse_categorical_crossentropy' is correct if your labels (y_train) are single integers (0, 1, 2, etc.))
model_bilstm.compile(loss='sparse_categorical_crossentropy',
                     optimizer='adam',
                     metrics=['accuracy'])

# stops training when the model stops improving to avoid overfitting (monitoring validation loss)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("starting model training...")

# train the model
history_bilstm = model_bilstm.fit(
    X_train_pad, y_train,
    validation_split=0.2, # use 20% of training data for validation during training
    epochs=10,          
    batch_size=256,
    callbacks=[early_stop]
)
print("training complete...")

Evaluation

In [ ]:
y_pred = np.argmax(model_bilstm.predict(X_test_pad), axis=1)

print("classification report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# confusion matrix
plt.figure(figsize=(12,8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=False, cmap="Greens")
plt.title("Confusion Matrix")
plt.show()

Usage

In [ ]:
def preprocess_text(text, tokenizer, max_len):
    # convert a single text into a padded sequence for the model.
    text_seq = tokenizer.texts_to_sequences([text])  # convert text to sequence
    text_pad = tf.keras.preprocessing.sequence.pad_sequences(
        text_seq, maxlen=max_len, padding='post'
    )
    return text_pad

# example text
new_text = "I am so happy and excited today!"

# preprocess it
input_seq = preprocess_text(new_text, tokenizer, max_len)

# predict
pred_probs = model_bilstm.predict(input_seq)  # returns probabilities
pred_class_index = pred_probs.argmax(axis=1)[0]  # index of max probability

# map back to emotion label
pred_emotion = label_encoder.inverse_transform([pred_class_index])[0]

print(f"Predicted emotion: {pred_emotion}")
print(f"Probabilities: {pred_probs[0]}")
